# Agent Platform Sessions サービスの仕組みを学ぶ

このノートブックでは、Agent Runtime にデプロイしたリソースのセッション情報を管理する Agent Platform Sessions サービスの仕組みを学びます。

## 事前準備

**[SST-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[SST-02]**

インストールされたパッケージのバージョンを確認します。

In [45]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[SST-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[SST-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [2]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[SST-05]**

この後の作業に必要なモジュールをインポートして、初期設定を行います。

In [3]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo
import agentplatform
from google.adk.sessions.vertex_ai_session_service import VertexAiSessionService

agentplatform.init(project=PROJECT_ID, location='us-central1')

**[SST-06]**

Agent Runtimeのクライアントオブジェクトを取得します。

In [4]:
agent_runtime = agentplatform.Client(location='us-central1').runtimes

**[SST-07]**

ノートブック「2. Agent Runtime Deployment.ipynb」で、Agent Runtimeにデプロイしたリソース（表示名 `Search Agent App`）のリソース名を確認して、これを操作するクライアントオブジェクトを取得します。

In [5]:
display_name = 'Search Agent App'

for item in agent_runtime.list():
    resource = item.api_resource
    if resource.display_name == display_name:
        resource_name = resource.name
        break

print(f'Resource Name for {display_name}: {resource_name}')
remote_adk_app = agent_runtime.get(name=resource_name)

Resource Name for Search Agent App: projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648


## セッション一覧の確認

**[SST-08]**

このリソースが Agent Platform Sessions サービスに保存したセッションの一覧を取得します。

ここでは、ノートブック「2. Agent Runtime Deployment.ipynb」で実行した会話に伴うセッションが得られます。

In [25]:
session_list = await remote_adk_app.async_list_sessions(
    user_id='default_user',
)
session_list

{'sessions': [{'appName': 'search_agent_app',
   'lastUpdateTime': 1788215665.87148,
   'state': {},
   'userId': 'default_user',
   'events': [],
   'id': '2024884837327831040'}]}

## セッション情報の取得

**[SST-09]**

上記で確認したセッションIDを指定して、セッションの内容を取得します。

In [27]:
session_id = session_list['sessions'][0]['id']

session = await remote_adk_app.async_get_session(
    user_id='default_user',
    session_id=session_id,
)

session.keys()

dict_keys(['userId', 'appName', 'id', 'lastUpdateTime', 'state', 'events'])

**[SST-10]**

得られたセッションの `events` 要素は、セッション内で発生したイベント（Event オブジェクト）のリストになります。

それぞれのイベントの `author` と `timestamp` を確認します。



In [21]:
for event in session['events']:
    author = event['author']
    timestamp = event['timestamp']
    print(f'{author}: {datetime.fromtimestamp(timestamp, tz=ZoneInfo('Asia/Tokyo'))}')

user: 2026-09-01 07:34:21.432259+09:00
search_agent: 2026-09-01 07:34:21.772988+09:00


この例では、2つのイベントが含まれており、それぞれ、ユーザーの入力に対応するイベントとそれに対する AI エージェントの応答に対応するイベントになります。

**[SST-11]**

最初のイベントの `content` 要素を表示すると、ユーザーの入力メッセージが確認できます。

In [22]:
session['events'][0]['content']

{'role': 'user',
 'parts': [{'videoMetadata': None,
   'functionCall': None,
   'codeExecutionResult': None,
   'thought': None,
   'mediaResolution': None,
   'thoughtSignature': None,
   'toolCall': None,
   'mediaProcessing': None,
   'text': '\n今日の新宿区の天気は？\n',
   'fileData': None,
   'audioTranscription': None,
   'functionResponse': None,
   'partMetadata': None,
   'executableCode': None,
   'toolResponse': None,
   'inlineData': None}]}

**[SST-12]**

２つ目のイベントの `content` 要素を表示すると、AIエージェントの応答メッセージが確認できます。

In [23]:
session['events'][1]['content']

{'role': 'model',
 'parts': [{'functionResponse': None,
   'fileData': None,
   'inlineData': None,
   'videoMetadata': None,
   'functionCall': None,
   'toolCall': None,
   'codeExecutionResult': None,
   'partMetadata': None,
   'text': '今日（9月1日）の新宿区は、朝は雲が広がっていますが、昼前からは日差しが届く見込みです。夕方以降は再び雲が多くなりますが、雨の心配はほとんどありません。\n\n* **天気**：曇りのち晴れ（日中はおおむね晴れ間が出ます）\n* **最高気温**：31℃前後\n* **服装**：真夏のような暑さや蒸し暑さが続くため、半袖など風通しのよい服装がおすすめです。熱中症対策や、室内外の温度差への備えを意識してお過ごしくださいね。',
   'toolResponse': None,
   'audioTranscription': None,
   'thought': None,
   'thoughtSignature': 'AY89a184onBCW9vfpZdMDlyQQ8-xq7AYiAvx3sVifJ6VE_LTdilwN-GoQhES_dLuD7Pps3GXQ-GTq1EDTh9MpbM1KAXN6WxfN3VZ9QkRl1bpRhM=',
   'executableCode': None,
   'mediaResolution': None,
   'mediaProcessing': None}]}

**[SST-13]**

AI エージェントの応答に対応する Event オブジェクトは、ノートブック「1. Agent Development Basics.ipynb」で確認した Event オブジェクトと同じもので、Google 検索に使用したキーワードなども同様に確認できます。

**注意**: Agent Platform Sessions サービスから取得したセッション情報は、ディクショナリのキーがキャメルケースになっているので注意してください。（ローカル環境では、スネークケースになっていました。）

In [24]:
session['events'][1]['groundingMetadata']['webSearchQueries']

['新宿区 天気 2026年9月1日', '新宿区 天気 今日', '東京都 9月1日 天気 気象庁']

## VertexAiSessionService の利用

**[SST-14]**

Agent Platform Sessions サービスの操作に使用する、VertexAiSessionService オブジェクトを生成します。

Agent Platform Sessions サービスは、複数のリージョンに用意されているので、使用する Agent Runtime と同じリージョン `location` オプションにを指定します。

In [33]:
session_service = VertexAiSessionService(location='us-central1')

**[SST-15]**

VertexAiSessionService オブジェクトは、該当リージョンに保存されたすべてのセッション情報にアクセスできるので、セッション一覧を取得する際は、リソース名とユーザー ID を指定します。

In [34]:
session_list = await session_service.list_sessions(
    app_name=resource_name,
    user_id='default_user',
)
session_list

ListSessionsResponse(sessions=[Session(id='2024884837327831040', app_name='projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648', user_id='default_user', state={}, events=[], last_update_time=1788215665.87148)])

上記の出力結果から、セッションを特定する、リソース名 `app_name`、ユーザー ID `user_id`、セッション ID `id` の三つ組が確認できます。

**[SST-16]**

確認したセッション ID を指定して、セッション情報を取得します。

In [14]:
session_id = session_list.sessions[0].id

session = await session_service.get_session(
    app_name=resource_name,
    user_id='default_user',
    session_id=session_id
)

**[SST-17]**

取得したセッションに含まれる最初の Event オブジェクトの `content` 要素を確認します。

In [15]:
session.events[0].content

Content(
  parts=[
    Part(
      text="""
今日の新宿区の天気は？
"""
    ),
  ],
  role='user'
)

**[SST-18]**

同じく、２番目の Event オブジェクトの `content` 要素を確認します。

In [16]:
session.events[1].content

Content(
  parts=[
    Part(
      text="""今日（9月1日）の新宿区は、朝は雲が広がっていますが、昼前からは日差しが届く見込みです。夕方以降は再び雲が多くなりますが、雨の心配はほとんどありません。

* **天気**：曇りのち晴れ（日中はおおむね晴れ間が出ます）
* **最高気温**：31℃前後
* **服装**：真夏のような暑さや蒸し暑さが続くため、半袖など風通しのよい服装がおすすめです。熱中症対策や、室内外の温度差への備えを意識してお過ごしくださいね。""",
      thought_signature=b'\x01\x8f=k_8\xa2pB[\xdb\xdf\xa5\x97L\x0e\\\x90C\xcf\xb1\xab\xb0\x18\x88\x0b\xf1\xde\xc5b|\x9e\x95\x13\xf2\xd3v)p7\xe1\xa8B\x11\x12\xfd\xd2\xee\x0f\xb3\xe9\xb3q\x97C\xe1\x93\xabQ\x03N\x1fL\xa5\xb35(\x05\xcd\xe9l_7uY\xf5\t\x11\x97V\xe9F\x13'
    ),
  ],
  role='model'
)